# Fine-tuning LLM Kecil untuk Deployment di Ollama

Notebook ini melakukan fine-tuning model kecil (Llama-3.2-3B-Instruct atau Qwen2.5-1.5B-Instruct) dengan LoRA menggunakan **Unsloth**, lalu meng-export hasilnya ke format **GGUF** supaya bisa langsung dipakai di **Ollama**.

**Sebelum mulai:** pastikan runtime Colab pakai GPU. Runtime > Change runtime type > T4 GPU.

**Langkah di notebook ini:**
1. Install dependencies
2. Load base model (4-bit quantized) dengan Unsloth
3. Setup LoRA adapter
4. Siapkan dataset (upload file `.jsonl`)
5. Training dengan SFTTrainer
6. Test inference cepat
7. Export ke GGUF untuk Ollama
8. Download hasil / simpan ke Google Drive

## 1. Install dependencies

In [1]:
%%capture
!pip install unsloth
# Selalu pakai versi terbaru unsloth (nightly) supaya kompatibel dengan Colab
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. Load base model (4-bit) dengan Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto detect, biarkan None
load_in_4bit = True

# Ganti model_name sesuai kebutuhan:
# - "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  (lebih besar, kualitas lebih baik)
# - "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"  (lebih ringan & cepat, cocok untuk waktu terbatas)
model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## 3. Setup LoRA adapter

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.9.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 4. Siapkan dataset

Dua opsi:
- **Opsi A**: Upload file `dataset.jsonl` sendiri (format Alpaca: `instruction`, `input`, `output`)
- **Opsi B**: Pakai dataset dummy contoh di bawah biar bisa langsung jalan tanpa upload apa-apa

Format tiap baris `.jsonl`:
```json
{"instruction": "Pertanyaan atau instruksi", "input": "", "output": "Jawaban"}
```

In [4]:
#Sambungkan Colab ke Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
dataset_path = "/content/drive/MyDrive/traffic_monitoring_qa.jsonl"

In [8]:
from datasets import load_dataset

alpaca_prompt = """Di bawah ini adalah instruksi yang menjelaskan sebuah tugas. Tulis jawaban yang sesuai untuk melengkapi permintaan tersebut.

### Instruksi:
{}

### Jawaban:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("json", data_files=dataset_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

Di bawah ini adalah instruksi yang menjelaskan sebuah tugas. Tulis jawaban yang sesuai untuk melengkapi permintaan tersebut.

### Instruksi:
Apa itu sistem traffic monitoring berbasis AI?

### Jawaban:
Sistem traffic monitoring berbasis AI adalah sistem yang menggunakan kecerdasan buatan untuk mendeteksi, melacak, menghitung, dan menganalisis kondisi lalu lintas dari data seperti video CCTV secara otomatis.<|im_end|>


## 5. Training dengan SFTTrainer

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,          #naikkan kalau dataset lebih besar
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/62 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 62 | Num Epochs = 8 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.423606
10,1.653347
15,1.244370
20,0.975926
25,0.758760
30,0.726155
35,0.599171
40,0.556403
45,0.479527
50,0.454278


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


## 6. Test inference cepat (sebelum export)

In [10]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [alpaca_prompt.format("Apa itu deteksi kepadatan lalu lintas?", "")],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Di bawah ini adalah instruksi yang menjelaskan sebuah tugas. Tulis jawaban yang sesuai untuk melengkapi permintaan tersebut.

### Instruksi:
Apa itu deteksi kepadatan lalu lintas?

### Jawaban:
Deteksi kepadatan lalu lintas adalah proses mengidentifikasi tingkat kepadatan kendaraan pada suatu area atau waktu tertentu dengan menggunakan estimasi jumlah kendaraan berdasarkan jumlah pixel atau proporsi ruas jalan yang dioccupy.


## 7. Export ke GGUF untuk Ollama

Unsloth punya fungsi bawaan untuk langsung convert ke GGUF dengan quantization (Q4_K_M direkomendasikan: ukuran kecil, kualitas masih bagus).

In [11]:
# Proses ini akan build llama.cpp otomatis di background, bisa makan waktu beberapa menit
model.save_pretrained_gguf("model-gguf", tokenizer, quantization_method = "q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in model-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:26<00:00, 26.29s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:04<00:00, 64.61s/it]


Unsloth: Merge process complete. Saved to `/content/model-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10840-mix-d5c17a0 (app-b10840-mix-d5c17a0-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model-gguf_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successf

{'save_directory': 'model-gguf',
 'gguf_directory': 'model-gguf_gguf',
 'gguf_files': ['model-gguf_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'model-gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

## 8. Download hasil GGUF

File GGUF akan ada di folder `model-gguf/`. Download file `.gguf`-nya, lalu pakai di Ollama dengan `Modelfile` seperti ini:

```
FROM ./Qwen2.5-1.5B-Instruct.Q4_K_M.gguf
PARAMETER temperature 0.7
SYSTEM "Kamu adalah asisten AI yang menjawab pertanyaan seputar sistem traffic monitoring."
```

Lalu jalankan di terminal lokal:
```bash
ollama create nama-model -f Modelfile
ollama run nama-model
```

In [16]:
import os

for f in os.listdir("model-gguf"):
    print(f)

# Untuk download otomatis dari Colab (opsional):
from google.colab import files
gguf_file = [f for f in os.listdir("model-gguf_gguf") if f.endswith(".gguf")][0]
files.download(f"model-gguf_gguf/{gguf_file}")

generation_config.json
model.safetensors
config.json
.cache
chat_template.jinja
tokenizer.json
tokenizer_config.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>